## Data Transformations with PySpark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('data_transformer').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/25 10:32:09 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/25 10:32:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 10:32:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!curl https://raw.githubusercontent.com/markumreed/data_science_for_everyone/refs/heads/main/pyspark_examples/data/fake_customers.csv >> customers.csv

!head customers.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   207  100   207    0     0    361      0 --:--:-- --:--:-- --:--:--   361
Name,Phone,Group
John,4085552424,A
Mike,3105552738,B
Cassie,4085552424,B
Laura,3105552438,B
Sarah,4085551234,A
David,3105557463,C
Zach,4085553987,C
Kiera,3105552938,A
Alexa,4085559467,C


In [5]:
df = spark.read.csv('customers.csv', inferSchema=True, header=True)

In [6]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- Group: string (nullable = true)



In [7]:
df.show()

+-------+----------+-----+
|   Name|     Phone|Group|
+-------+----------+-----+
|   John|4085552424|    A|
|   Mike|3105552738|    B|
| Cassie|4085552424|    B|
|  Laura|3105552438|    B|
|  Sarah|4085551234|    A|
|  David|3105557463|    C|
|   Zach|4085553987|    C|
|  Kiera|3105552938|    A|
|  Alexa|4085559467|    C|
|Karissa|3105553475|    A|
+-------+----------+-----+



### Data Features

#### StringIndexer

* Convert string data into numerical (categorical feature)
* Encode as dummy variables/OneHotEncoder
* `StringIndexer`

In [9]:
from pyspark.ml.feature import StringIndexer

df2 = spark.createDataFrame(
    [(0, 'a'), (1, 'b'), (2, 'c'), (3, 'a'), (4, 'b'), (5, 'c')],
    ['user_id', 'category']
)

In [10]:
df2.show()

+-------+--------+
|user_id|category|
+-------+--------+
|      0|       a|
|      1|       b|
|      2|       c|
|      3|       a|
|      4|       b|
|      5|       c|
+-------+--------+



#### VectorIndexer

* **VectorAssembler** is a transformer that combines a given list of columns into a single vector column.
* **VectorAssembler** accepts the following input column types:

  * all numeric types, boolean type, and vector type.

In [11]:
indexer = StringIndexer(inputCol='category', outputCol='categoryIndex')

In [12]:
indexed = indexer.fit(df2).transform(df2)

In [13]:
indexed.show()

+-------+--------+-------------+
|user_id|category|categoryIndex|
+-------+--------+-------------+
|      0|       a|          0.0|
|      1|       b|          1.0|
|      2|       c|          2.0|
|      3|       a|          0.0|
|      4|       b|          1.0|
|      5|       c|          2.0|
+-------+--------+-------------+



In [17]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

In [18]:
df3 = spark.createDataFrame(
    [(0, 18, 1.0, Vectors.dense([0.0, 10.0, 0.5]), 1.0)],
    ['id', 'hour', 'mobile', 'userFeatures', 'clicked']
)
df3.show()

+---+----+------+--------------+-------+
| id|hour|mobile|  userFeatures|clicked|
+---+----+------+--------------+-------+
|  0|  18|   1.0|[0.0,10.0,0.5]|    1.0|
+---+----+------+--------------+-------+



In [19]:
assembler = VectorAssembler(
    inputCols = ['hour', 'mobile', 'userFeatures'],
    outputCol = 'features'
)

output = assembler.transform(df3)

In [20]:
output.show()

+---+----+------+--------------+-------+--------------------+
| id|hour|mobile|  userFeatures|clicked|            features|
+---+----+------+--------------+-------+--------------------+
|  0|  18|   1.0|[0.0,10.0,0.5]|    1.0|[18.0,1.0,0.0,10....|
+---+----+------+--------------+-------+--------------------+



In [21]:
output.select('features', 'clicked').show()

+--------------------+-------+
|            features|clicked|
+--------------------+-------+
|[18.0,1.0,0.0,10....|    1.0|
+--------------------+-------+



#### Example with Customer Data

In [22]:
df.show()

+-------+----------+-----+
|   Name|     Phone|Group|
+-------+----------+-----+
|   John|4085552424|    A|
|   Mike|3105552738|    B|
| Cassie|4085552424|    B|
|  Laura|3105552438|    B|
|  Sarah|4085551234|    A|
|  David|3105557463|    C|
|   Zach|4085553987|    C|
|  Kiera|3105552938|    A|
|  Alexa|4085559467|    C|
|Karissa|3105553475|    A|
+-------+----------+-----+



In [23]:
indexer = StringIndexer(inputCol='Group', outputCol='groupindex')
indexed = indexer.fit(df).transform(df)
indexed.show()

+-------+----------+-----+----------+
|   Name|     Phone|Group|groupindex|
+-------+----------+-----+----------+
|   John|4085552424|    A|       0.0|
|   Mike|3105552738|    B|       1.0|
| Cassie|4085552424|    B|       1.0|
|  Laura|3105552438|    B|       1.0|
|  Sarah|4085551234|    A|       0.0|
|  David|3105557463|    C|       2.0|
|   Zach|4085553987|    C|       2.0|
|  Kiera|3105552938|    A|       0.0|
|  Alexa|4085559467|    C|       2.0|
|Karissa|3105553475|    A|       0.0|
+-------+----------+-----+----------+



In [26]:
assembler = VectorAssembler(
    inputCols = ['Phone', 'groupindex'],
    outputCol = 'features'
)

output = assembler.transform(indexed)

output.show()

+-------+----------+-----+----------+-------------------+
|   Name|     Phone|Group|groupindex|           features|
+-------+----------+-----+----------+-------------------+
|   John|4085552424|    A|       0.0|[4.085552424E9,0.0]|
|   Mike|3105552738|    B|       1.0|[3.105552738E9,1.0]|
| Cassie|4085552424|    B|       1.0|[4.085552424E9,1.0]|
|  Laura|3105552438|    B|       1.0|[3.105552438E9,1.0]|
|  Sarah|4085551234|    A|       0.0|[4.085551234E9,0.0]|
|  David|3105557463|    C|       2.0|[3.105557463E9,2.0]|
|   Zach|4085553987|    C|       2.0|[4.085553987E9,2.0]|
|  Kiera|3105552938|    A|       0.0|[3.105552938E9,0.0]|
|  Alexa|4085559467|    C|       2.0|[4.085559467E9,2.0]|
|Karissa|3105553475|    A|       0.0|[3.105553475E9,0.0]|
+-------+----------+-----+----------+-------------------+



In [27]:
output.select('Name', 'features').show()

+-------+-------------------+
|   Name|           features|
+-------+-------------------+
|   John|[4.085552424E9,0.0]|
|   Mike|[3.105552738E9,1.0]|
| Cassie|[4.085552424E9,1.0]|
|  Laura|[3.105552438E9,1.0]|
|  Sarah|[4.085551234E9,0.0]|
|  David|[3.105557463E9,2.0]|
|   Zach|[4.085553987E9,2.0]|
|  Kiera|[3.105552938E9,0.0]|
|  Alexa|[4.085559467E9,2.0]|
|Karissa|[3.105553475E9,0.0]|
+-------+-------------------+

